In [17]:
import os
from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain


In [18]:

# 1. Configure OpenSearch Connection Details
OPENSEARCH_URL = "http://localhost:9200"
INDEX_NAME = "my-ollama-rag-index"
HTTP_AUTH = ("admin", "admin")  # Update with your credentials if required


In [19]:

# 2. Reinitialize the identical embedding model used during ingestion
# (Crucial: Must be the exact same model to ensure vector dimensions match)
embedding_model = OllamaEmbeddings(model="nomic-embed-text")


In [20]:
# 3. Connect to the existing OpenSearch Vector Index
vector_store = OpenSearchVectorSearch(
    opensearch_url=OPENSEARCH_URL,
    index_name=INDEX_NAME,
    embedding_function=embedding_model,
    http_auth=HTTP_AUTH,
    use_ssl=False,
    verify_certs=False
)


In [21]:
# 4. Convert the vector store into a LangChain Retriever
# This configures it to fetch the top 3 most relevant chunks (k=3)
# Force the search type to standard "similarity"
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [22]:

# 5. Initialize the ChatOllama Model for generating answers
llm = ChatOllama(model="llama3", temperature=0)


In [23]:


# 6. Define a systematic prompt for the RAG workflow
system_prompt = (
    "You are an expert assistant tasking with answering questions.\n"
    "Use the following pieces of retrieved context to answer the question. "
    "If you do not know the answer, say that you do not know.\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


In [24]:

# 7. Assemble the Retrieval-QA Chain
# Combine documents chain injects the retrieved context chunks into the prompt
question_answer_chain = create_stuff_documents_chain(llm, prompt)
# Retrieval chain coordinates fetching from OpenSearch and running the LLM
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# 8. Execute a user query
user_query = "What is OpenSearch capable of handling?"


In [25]:


print(f"Sending query: '{user_query}'...\n")
response = rag_chain.invoke({"input": user_query})


Sending query: 'What is OpenSearch capable of handling?'...



RequestError: RequestError(400, 'parsing_exception', 'unknown query [knn]')

In [ ]:

# 9. Print results
print("--- ANSWER ---")
print(response["answer"])

print("\n--- RETRIEVED SOURCE DOCUMENTS ---")
for i, doc in enumerate(response["context"]):
    print(f"\n[Chunk {i+1}] Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content: {doc.page_content.strip()}")
